# Capítulo 10 - Projeto: Circuito RC

**Disciplina:** FSC1189 - Algoritmo e Programação

**Tema:** Análise de Circuito RC usando simulação numérica

Este notebook implementa a solução analítica e numérica (método de Euler) para um circuito RC em carga e descarga.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.integrate import odeint

## Parte 1: Fundamentos de Circuito RC

Um circuito RC é composto por um resistor (R) e um capacitor (C) em série.

**Equação diferencial:**
$$V_C + V_R = V_{bateria}$$
$$V_C + R\frac{dQ}{dt} = V_0$$

Como $Q = CV_C$, temos $\frac{dQ}{dt} = C\frac{dV_C}{dt}$

$$\frac{dV_C}{dt} + \frac{V_C}{RC} = \frac{V_0}{RC}$$

**Solução analítica (carga):**
$$V_C(t) = V_0(1 - e^{-t/\tau})$$

onde $\tau = RC$ é a constante de tempo.

**Solução analítica (descarga):**
$$V_C(t) = V_0 e^{-t/\tau}$$

In [ ]:
# Parâmetros do circuito
R = 1000  # Ohms
C = 1e-6  # Farads (1 µF)
tau = R * C  # Constante de tempo (segundos)
V0 = 10  # Tensão da bateria (Volts)

print("=== CIRCUITO RC ===")
print(f"Resistência: R = {R} Ω")
print(f"Capacitância: C = {C*1e6} µF")
print(f"Constante de tempo: τ = RC = {tau:.3f} s")
print(f"Tensão da bateria: V₀ = {V0} V")
print()
print(f"Tempo para atingir 63.2% (1τ): {tau:.4f} s")
print(f"Tempo para atingir 95% (3τ): {3*tau:.4f} s")
print(f"Tempo para atingir 99% (5τ): {5*tau:.4f} s")

## Parte 2: Soluções Analítica e Numérica (Método de Euler)

In [ ]:
def vc_carga_analitica(t, tau, V0):
    """Solução analítica para carga do capacitor."""
    return V0 * (1 - np.exp(-t / tau))

def vc_descarga_analitica(t, tau, V0):
    """Solução analítica para descarga do capacitor."""
    return V0 * np.exp(-t / tau)

def metodo_euler_carga(V0, R, C, dt, t_max):
    """Simula carga do capacitor usando método de Euler.
    
    EDO: dVc/dt = (V0 - Vc) / (RC)
    """
    tau = R * C
    t_array = np.arange(0, t_max + dt, dt)
    Vc = np.zeros_like(t_array)
    Vc[0] = 0
    
    for i in range(len(t_array) - 1):
        dVc_dt = (V0 - Vc[i]) / tau
        Vc[i + 1] = Vc[i] + dVc_dt * dt
    
    return t_array, Vc

def metodo_euler_descarga(V0, R, C, dt, t_max):
    """Simula descarga do capacitor usando método de Euler.
    
    EDO: dVc/dt = -Vc / (RC)
    """
    tau = R * C
    t_array = np.arange(0, t_max + dt, dt)
    Vc = np.zeros_like(t_array)
    Vc[0] = V0  # Capacitor começa carregado
    
    for i in range(len(t_array) - 1):
        dVc_dt = -Vc[i] / tau
        Vc[i + 1] = Vc[i] + dVc_dt * dt
    
    return t_array, Vc

# Simular
t_total = 5 * tau  # 5 constantes de tempo
dt = tau / 100  # Passo de tempo pequeno

# Solução analítica
t_analitico = np.linspace(0, t_total, 500)
Vc_carga_anal = vc_carga_analitica(t_analitico, tau, V0)
Vc_descarga_anal = vc_descarga_analitica(t_analitico, tau, V0)

# Solução numérica
t_euler_carga, Vc_euler_carga = metodo_euler_carga(V0, R, C, dt, t_total)
t_euler_descarga, Vc_euler_descarga = metodo_euler_descarga(V0, R, C, dt, t_total)

# Calcular erros
erro_carga = np.interp(t_euler_carga, t_analitico, Vc_carga_anal) - Vc_euler_carga
erro_descarga = np.interp(t_euler_descarga, t_analitico, Vc_descarga_anal) - Vc_euler_descarga

print("=== SIMULAÇÕES REALIZADAS ===")
print(f"Tempo total de simulação: {t_total:.4f} s")
print(f"Passo de tempo (Euler): dt = {dt:.6f} s")
print(f"Número de passos (carga): {len(t_euler_carga)}")
print()
print(f"Erro máximo em carga: {np.max(np.abs(erro_carga)):.2e} V")
print(f"Erro máximo em descarga: {np.max(np.abs(erro_descarga)):.2e} V")

## Parte 3: Visualizações

In [ ]:
# Criar figura com 4 painéis
fig = plt.figure(figsize=(14, 10))
gs = fig.add_gridspec(2, 2, hspace=0.3, wspace=0.3)

# Painel 1: Carga - Analítico vs Euler
ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(t_analitico, Vc_carga_anal, 'b-', linewidth=2.5, label='Analítico')
ax1.plot(t_euler_carga, Vc_euler_carga, 'r--', linewidth=1.5, alpha=0.7, label='Euler')
ax1.axhline(0.632 * V0, color='g', linestyle=':', alpha=0.5, label='63.2% (1τ)')
ax1.axvline(tau, color='g', linestyle=':', alpha=0.5)
ax1.set_xlabel('Tempo (s)', fontsize=10)
ax1.set_ylabel('Vc (V)', fontsize=10)
ax1.set_title('Carga do Capacitor', fontsize=11, fontweight='bold')
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)
ax1.set_xlim(0, t_total)

# Painel 2: Descarga - Analítico vs Euler
ax2 = fig.add_subplot(gs[0, 1])
ax2.plot(t_analitico, Vc_descarga_anal, 'b-', linewidth=2.5, label='Analítico')
ax2.plot(t_euler_descarga, Vc_euler_descarga, 'r--', linewidth=1.5, alpha=0.7, label='Euler')
ax2.axhline(0.368 * V0, color='g', linestyle=':', alpha=0.5, label='36.8% (1τ)')
ax2.axvline(tau, color='g', linestyle=':', alpha=0.5)
ax2.set_xlabel('Tempo (s)', fontsize=10)
ax2.set_ylabel('Vc (V)', fontsize=10)
ax2.set_title('Descarga do Capacitor', fontsize=11, fontweight='bold')
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)
ax2.set_xlim(0, t_total)

# Painel 3: Corrente durante carga
ax3 = fig.add_subplot(gs[1, 0])
I_carga = (V0 - np.interp(t_euler_carga, t_analitico, Vc_carga_anal)) / R
I_descarga = -np.interp(t_euler_descarga, t_analitico, Vc_descarga_anal) / R

ax3.plot(t_euler_carga, I_carga * 1e3, 'purple', linewidth=2, label='Carga')
ax3.plot(t_euler_descarga, I_descarga * 1e3, 'orange', linewidth=2, label='Descarga')
ax3.axhline(V0/R * 1e3, color='gray', linestyle='--', alpha=0.5, label='I₀')
ax3.set_xlabel('Tempo (s)', fontsize=10)
ax3.set_ylabel('Corrente (mA)', fontsize=10)
ax3.set_title('Corrente no Circuito', fontsize=11, fontweight='bold')
ax3.legend(fontsize=9)
ax3.grid(True, alpha=0.3)
ax3.set_xlim(0, t_total)

# Painel 4: Energia no capacitor
ax4 = fig.add_subplot(gs[1, 1])
E_cap_carga = 0.5 * C * Vc_euler_carga**2 * 1e6  # em µJ
E_cap_descarga = 0.5 * C * Vc_euler_descarga**2 * 1e6

ax4.plot(t_euler_carga, E_cap_carga, 'blue', linewidth=2, label='Carga')
ax4.plot(t_euler_descarga, E_cap_descarga, 'orange', linewidth=2, label='Descarga')
ax4.axhline(0.5 * C * V0**2 * 1e6 * 0.632, color='g', linestyle=':', alpha=0.5, label='63.2% E₀ (1τ)')
ax4.set_xlabel('Tempo (s)', fontsize=10)
ax4.set_ylabel('Energia (µJ)', fontsize=10)
ax4.set_title('Energia Armazenada no Capacitor', fontsize=11, fontweight='bold')
ax4.legend(fontsize=9)
ax4.grid(True, alpha=0.3)
ax4.set_xlim(0, t_total)

plt.suptitle('Análise de Circuito RC: Carga e Descarga', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Energia total
E_total = 0.5 * C * V0**2
print(f"\nEnergia máxima do capacitor: {E_total*1e6:.3f} µJ")

## Parte 4: Efeito da Resistência sobre τ

Visualizar como a constante de tempo muda com diferentes valores de R.

In [ ]:
# Variar resistência
R_valores = np.array([100, 500, 1000, 5000, 10000])  # Ohms
taus = R_valores * C

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Painel 1: Comparação de cargas para diferentes R
ax = axes[0]
for R_val, tau_val in zip(R_valores, taus):
    Vc = vc_carga_analitica(t_analitico, tau_val, V0)
    ax.plot(t_analitico, Vc, linewidth=2.5, label=f'R={R_val}Ω (τ={tau_val*1e3:.2f}ms)')

ax.set_xlabel('Tempo (s)', fontsize=11)
ax.set_ylabel('Vc (V)', fontsize=11)
ax.set_title('Carga do Capacitor para Diferentes Resistências', fontsize=12, fontweight='bold')
ax.legend(fontsize=10, loc='right')
ax.grid(True, alpha=0.3)
ax.set_xlim(0, t_total)

# Painel 2: Constante de tempo vs Resistência
ax = axes[1]
ax.plot(R_valores, taus * 1e3, 'b-o', linewidth=2.5, markersize=8)
ax.fill_between(R_valores, 0, taus * 1e3, alpha=0.2)
ax.set_xlabel('Resistência R (Ω)', fontsize=11)
ax.set_ylabel('Constante de Tempo τ (ms)', fontsize=11)
ax.set_title(f'Relação τ = RC (C = {C*1e6}µF)', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("=== CONSTANTE DE TEMPO PARA DIFERENTES R ===")
for R_val, tau_val in zip(R_valores, taus):
    print(f"R = {R_val:5}Ω  →  τ = {tau_val*1e3:7.2f} ms")

## Exercício Proposto: Comparação de Métodos Numéricos

**Desafio:**

Implemente o método de Runge-Kutta de 4ª ordem (RK4) e compare sua precisão com o método de Euler para simular a carga do capacitor. 

Dica: Use `scipy.integrate.odeint` que já implementa RK4 internamente.

In [ ]:
def modelo_rc(Vc, t, V0, tau):
    """EDO para circuito RC: dVc/dt = (V0 - Vc) / tau"""
    return (V0 - Vc) / tau

# Usar odeint (usa RK45 internamente)
t_rk45 = t_analitico
Vc_rk45 = odeint(modelo_rc, 0, t_rk45, args=(V0, tau)).flatten()

# Comparar erros: Euler vs RK45
Vc_euler_interp = np.interp(t_rk45, t_euler_carga, Vc_euler_carga)
erro_euler = np.abs(Vc_euler_interp - Vc_carga_anal)
erro_rk45 = np.abs(Vc_rk45 - Vc_carga_anal)

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.semilogy(t_analitico, erro_euler, 'r-', linewidth=2, label='Euler')
plt.semilogy(t_analitico, erro_rk45, 'g-', linewidth=2, label='RK45 (odeint)')
plt.xlabel('Tempo (s)', fontsize=11)
plt.ylabel('Erro Absoluto (V) - escala log', fontsize=11)
plt.title('Comparação de Erros: Euler vs RK45', fontsize=12, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3, which='both')

plt.subplot(1, 2, 2)
plt.plot(t_analitico, Vc_carga_anal, 'k--', linewidth=2.5, label='Analítico')
plt.plot(t_euler_carga, Vc_euler_carga, 'r-', linewidth=1, alpha=0.6, label='Euler')
plt.plot(t_rk45, Vc_rk45, 'g:', linewidth=2, label='RK45')
plt.xlabel('Tempo (s)', fontsize=11)
plt.ylabel('Vc (V)', fontsize=11)
plt.title('Soluções Numéricas vs Analítica', fontsize=12, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("=== COMPARAÇÃO DE PRECISÃO ===")
print(f"Erro máximo (Euler): {np.max(erro_euler):.2e} V")
print(f"Erro máximo (RK45): {np.max(erro_rk45):.2e} V")
print(f"Redução de erro: {np.max(erro_euler) / np.max(erro_rk45):.1f}x")